# Energy Analyst Programming Exercise — working notebook
Work happens here first; approved code is exported to `main.py`.

## Parameters
System-specific parameters (file paths, packages). Update these to run on another machine.

In [1]:
# Required packages: pandas, numpy, matplotlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()                          # repo root (main.py uses its own folder)
DATA_DIR = BASE_DIR / "historicalPriceData"   # historical power price data
OUTPUT_DIR = BASE_DIR / "output"              # all generated CSVs and images
OUTPUT_DIR.mkdir(exist_ok=True)

## Task 1 — Read and combine all historical data files

In [2]:
# Read every yearly price file and stack them into one dataframe
files = sorted(DATA_DIR.glob("*.csv"))
prices = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Parse timestamps so we can group by year/month and sort by hour later
prices["Date"] = pd.to_datetime(prices["Date"])

# Quick checks
print("Files read:", [f.name for f in files])
print("Rows:", len(prices))
print("Date range:", prices["Date"].min(), "to", prices["Date"].max())
print("Settlement points:", prices["SettlementPoint"].nunique())
print(sorted(prices["SettlementPoint"].unique()))
print("Missing prices:", prices["Price"].isna().sum())
prices.head()

Files read: ['ERCOT_DA_Prices_2016.csv', 'ERCOT_DA_Prices_2017.csv', 'ERCOT_DA_Prices_2018.csv', 'ERCOT_DA_Prices_2019.csv']
Rows: 497320
Date range: 2016-01-01 00:00:00 to 2019-12-31 23:00:00
Settlement points: 15
['HB_BUSAVG', 'HB_HOUSTON', 'HB_HUBAVG', 'HB_NORTH', 'HB_PAN', 'HB_SOUTH', 'HB_WEST', 'LZ_AEN', 'LZ_CPS', 'LZ_HOUSTON', 'LZ_LCRA', 'LZ_NORTH', 'LZ_RAYBN', 'LZ_SOUTH', 'LZ_WEST']
Missing prices: 0


,Date,SettlementPoint,Price
0,2016-01-01,HB_BUSAVG,18.42
1,2016-01-01,HB_HOUSTON,18.42
2,2016-01-01,HB_HUBAVG,18.42
3,2016-01-01,HB_NORTH,18.43
4,2016-01-01,HB_SOUTH,18.38


## Task 2 — Monthly average price per settlement point (Jan 2016–Dec 2019, keep prices ≤ 0)

In [6]:
# Add Year and Month columns from the hourly timestamp
prices["Year"] = prices["Date"].dt.year
prices["Month"] = prices["Date"].dt.month

# Average price per settlement point per year-month.
# Prices <= 0 are kept on purpose: negative prices are real in deregulated markets.
monthly_avg = (
    prices.groupby(["SettlementPoint", "Year", "Month"], as_index=False)["Price"]
    .mean()
    .rename(columns={"Price": "AveragePrice"})
)

# Quick checks
print("Rows:", len(monthly_avg))
print("Year-months per settlement point:")
print(monthly_avg.groupby("SettlementPoint").size().to_string())
print("Lowest monthly average:\n", monthly_avg.nsmallest(1, "AveragePrice").to_string(index=False))
print("Highest monthly average:\n", monthly_avg.nlargest(1, "AveragePrice").to_string(index=False))
monthly_avg.head()

Rows: 681
Year-months per settlement point:
SettlementPoint
HB_BUSAVG     48
HB_HOUSTON    48
HB_HUBAVG     48
HB_NORTH      48
HB_PAN         9
HB_SOUTH      48
HB_WEST       48
LZ_AEN        48
LZ_CPS        48
LZ_HOUSTON    48
LZ_LCRA       48
LZ_NORTH      48
LZ_RAYBN      48
LZ_SOUTH      48
LZ_WEST       48
Lowest monthly average:
 SettlementPoint  Year  Month  AveragePrice
         HB_PAN  2019     12      8.817245
Highest monthly average:
 SettlementPoint  Year  Month  AveragePrice
        LZ_WEST  2019      8    128.876465


,SettlementPoint,Year,Month,AveragePrice
0,HB_BUSAVG,2016,1,19.209301
1,HB_BUSAVG,2016,2,15.288247
2,HB_BUSAVG,2016,3,16.755626
3,HB_BUSAVG,2016,4,18.289347
4,HB_BUSAVG,2016,5,18.159583


## Task 3 — Write AveragePriceByMonth.csv (SettlementPoint, Year, Month, AveragePrice)

## Task 4 — Hourly volatility per year, hubs (HB_) only, drop prices ≤ 0 before log returns

## Task 5 — Write HourlyVolatilityByYear.csv (SettlementPoint, Year, HourlyVolatility)

## Task 6 — Highest-volatility hub per year → MaxVolatilityByYear.csv

## Task 7 — Model-ready files: output/formattedSpotHistory/spot_<SettlementPoint>.csv (15 files)

## Bonus — Mean plots: SettlementHubAveragePriceByMonth.png, LoadZoneAveragePriceByMonth.png

## Bonus — Volatility plots

## Bonus — Hourly shape profiles: output/hourlyShapeProfiles/profile_<SettlementPoint>.csv (15 files)

## Bonus — Open-ended analysis